In [1]:
%pip install -q azure-cognitiveservices-speech

In [2]:
from google.colab import userdata
import azure.cognitiveservices.speech as speechsdk

SPEECH_KEY = userdata.get("SPEECH_KEY")
SPEECH_REGION = userdata.get("SPEECH_REGION") or "eastus"

speech_config = speechsdk.SpeechConfig(
    subscription=SPEECH_KEY,
    region=SPEECH_REGION
)

speech_config.speech_synthesis_voice_name = "pt-BR-FranciscaNeural"

In [3]:
import os
import random
import html
import time
import pandas as pd

random.seed(42)

PASTA_ORIGINAL = "/content/dataset_audio_emocoes"
os.makedirs(PASTA_ORIGINAL, exist_ok=True)

classes = {
    "ansiedade_hesitacao": {
        "rate": "-10%",
        "pitch": "+10%",
        "volume": "default"
    },
    "neutro": {
        "rate": "0%",
        "pitch": "0%",
        "volume": "default"
    },
    "tristeza": {
        "rate": "-20%",
        "pitch": "-10%",
        "volume": "soft"
    }
}

frases = [
    "Estou me sentindo bem hoje",
    "Estou um pouco preocupado com o procedimento",
    "Não tenho certeza se isso está normal",
    "Estou sentindo um desconforto",
    "Acho que consigo continuar",
    "Estou com medo de que algo aconteça",
    "Minha respiração está normal",
    "Estou me sentindo cansado",
    "Preciso de ajuda para ficar mais tranquilo",
    "Estou com dificuldade para relaxar",
    "Sinto uma pressão no peito",
    "Estou tentando manter a calma",
    "Não sei explicar exatamente o que estou sentindo",
    "Estou muito nervoso neste momento",
    "A dor está diminuindo",
    "Estou me sentindo triste",
    "Estou sem disposição hoje",
    "Não estou me sentindo muito bem",
    "Acho que preciso descansar",
    "Está tudo bem, podemos continuar"
]

prefixos = {
    "ansiedade_hesitacao": [
        "",
        "Bem, ",
        "Hum, ",
        "Eu acho que ",
        "Talvez "
    ],
    "neutro": [
        "",
        "Certo, ",
        "Sim, ",
        "Agora, ",
        "Neste momento, "
    ],
    "tristeza": [
        "",
        "Infelizmente, ",
        "Bom, ",
        "Neste momento, ",
        "Eu sinto que "
    ]
}

In [4]:
registros = []

for classe, estilo in classes.items():

    combinacoes = [
        prefixo + frase
        for frase in frases
        for prefixo in prefixos[classe]
    ]

    random.shuffle(combinacoes)
    combinacoes = combinacoes[:100]

    for indice, texto in enumerate(combinacoes):

        texto_ssml = html.escape(texto)

        if classe == "ansiedade_hesitacao":
            texto_ssml = texto_ssml.replace(
                "acho que",
                "acho que <break time='400ms'/>"
            ).replace(
                "Talvez",
                "Talvez <break time='500ms'/>"
            )

        elif classe == "tristeza":
            texto_ssml = texto_ssml.replace(
                " ",
                " <break time='100ms'/> ",
                2
            )

        ssml = f"""
        <speak version="1.0"
               xmlns="http://www.w3.org/2001/10/synthesis"
               xml:lang="pt-BR">
            <voice name="pt-BR-FranciscaNeural">
                <prosody rate="{estilo['rate']}"
                         pitch="{estilo['pitch']}"
                         volume="{estilo['volume']}">
                    {texto_ssml}
                </prosody>
            </voice>
        </speak>
        """

        nome = f"{classe}_{indice:03d}.wav"
        caminho = os.path.join(PASTA_ORIGINAL, nome)

        audio_config = speechsdk.audio.AudioOutputConfig(
            filename=caminho
        )

        sintetizador = speechsdk.SpeechSynthesizer(
            speech_config=speech_config,
            audio_config=audio_config
        )

        resultado = sintetizador.speak_ssml_async(ssml).get()

        if resultado.reason == speechsdk.ResultReason.SynthesizingAudioCompleted:
            tamanho = os.path.getsize(caminho)

            if tamanho > 0:
                registros.append({
                    "arquivo": nome,
                    "caminho": caminho,
                    "classe": classe,
                    "texto": texto,
                    "origem": "azure_speech_sintetico"
                })

                print("OK:", nome)

        else:
            detalhes = resultado.cancellation_details
            print("ERRO:", nome)
            print("Motivo:", detalhes.reason)
            print("Código:", detalhes.error_code)
            print("Detalhes:", detalhes.error_details)

        time.sleep(0.3)

df_original = pd.DataFrame(registros)

df_original.to_csv(
    os.path.join(PASTA_ORIGINAL, "metadata_original.csv"),
    index=False,
    encoding="utf-8"
)

print(df_original["classe"].value_counts())

OK: ansiedade_hesitacao_000.wav
OK: ansiedade_hesitacao_001.wav
OK: ansiedade_hesitacao_002.wav
OK: ansiedade_hesitacao_003.wav
OK: ansiedade_hesitacao_004.wav
OK: ansiedade_hesitacao_005.wav
OK: ansiedade_hesitacao_006.wav
OK: ansiedade_hesitacao_007.wav
OK: ansiedade_hesitacao_008.wav
OK: ansiedade_hesitacao_009.wav
OK: ansiedade_hesitacao_010.wav
OK: ansiedade_hesitacao_011.wav
OK: ansiedade_hesitacao_012.wav
OK: ansiedade_hesitacao_013.wav
OK: ansiedade_hesitacao_014.wav
OK: ansiedade_hesitacao_015.wav
OK: ansiedade_hesitacao_016.wav
OK: ansiedade_hesitacao_017.wav
OK: ansiedade_hesitacao_018.wav
OK: ansiedade_hesitacao_019.wav
OK: ansiedade_hesitacao_020.wav
OK: ansiedade_hesitacao_021.wav
OK: ansiedade_hesitacao_022.wav
OK: ansiedade_hesitacao_023.wav
OK: ansiedade_hesitacao_024.wav
OK: ansiedade_hesitacao_025.wav
OK: ansiedade_hesitacao_026.wav
OK: ansiedade_hesitacao_027.wav
OK: ansiedade_hesitacao_028.wav
OK: ansiedade_hesitacao_029.wav
OK: ansiedade_hesitacao_030.wav
OK: ansi

In [5]:
!pip -q install librosa soundfile

In [12]:
import librosa
import soundfile as sf
import numpy as np
import shutil

PASTA_FINAL = "/content/dataset_audio_emocoes_1200"
os.makedirs(PASTA_FINAL, exist_ok=True)

registros_finais = []

for _, linha in df_original.iterrows():

    audio, sr = librosa.load(
        linha["caminho"],
        sr=16000,
        mono=True
    )

    base = os.path.splitext(linha["arquivo"])[0]

    variacoes = {
        "original": audio,
        "lento": librosa.effects.time_stretch(
            audio,
            rate=0.92
        ),
        "rapido": librosa.effects.time_stretch(
            audio,
            rate=1.08
        ),
        "ruido": audio + np.random.normal(
            0,
            0.003,
            len(audio)
        )
    }

    for tipo, audio_variado in variacoes.items():

        audio_variado = np.clip(audio_variado, -1, 1)

        nome = f"{base}_{tipo}.wav"
        caminho = os.path.join(PASTA_FINAL, nome)

        sf.write(caminho, audio_variado, sr)

        registros_finais.append({
            "arquivo": nome,
            "caminho": caminho,
            "classe": linha["classe"],
            "texto": linha["texto"],
            "arquivo_original": linha["arquivo"],
            "tipo": tipo,
            "origem": "sintetico_azure_com_aumento_local"
        })

df_final = pd.DataFrame(registros_finais)

df_final.to_csv(
    os.path.join(PASTA_FINAL, "metadata.csv"),
    index=False,
    encoding="utf-8"
)

print(df_final["classe"].value_counts())
print("Total:", len(df_final))

classe
ansiedade_hesitacao    400
neutro                 400
tristeza               400
Name: count, dtype: int64
Total: 1200


In [13]:
import shutil
from google.colab import files

arquivo_zip = shutil.make_archive(
    "/content/dataset_audio_emocoes_1200",
    "zip",
    PASTA_FINAL
)

files.download(arquivo_zip)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
%pip install -q openai-whisper librosa soundfile scikit-learn tqdm

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [15]:
import whisper
import librosa
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

modelo_whisper = whisper.load_model("tiny")

df_meta = pd.read_csv(
    "/content/dataset_audio_emocoes_1200/metadata.csv"
)

resultados = []

for _, linha in tqdm(
    df_meta.iterrows(),
    total=len(df_meta)
):
    arquivo = linha["caminho"]

    audio, sr = librosa.load(
        arquivo,
        sr=16000,
        mono=True
    )

    transcricao = modelo_whisper.transcribe(
        arquivo,
        language="pt",
        fp16=False
    )

    texto = transcricao["text"].strip()
    quantidade_palavras = len(texto.split())

    duracao = librosa.get_duration(
        y=audio,
        sr=sr
    )

    rms = librosa.feature.rms(y=audio)[0]
    energia_media = float(np.mean(rms))

    intervalos = librosa.effects.split(
        audio,
        top_db=30
    )

    tempo_sonoro = sum(
        (fim - inicio) / sr
        for inicio, fim in intervalos
    )

    tempo_pausas = max(duracao - tempo_sonoro, 0)
    proporcao_pausas = tempo_pausas / duracao

    ritmo = (
        quantidade_palavras / (duracao / 60)
        if duracao > 0 else 0
    )

    f0, _, _ = librosa.pyin(
        audio,
        fmin=70,
        fmax=500,
        sr=sr
    )

    f0_valido = f0[~np.isnan(f0)]

    tom_medio = (
        float(np.mean(f0_valido))
        if len(f0_valido) > 0 else 0
    )

    tom_desvio = (
        float(np.std(f0_valido))
        if len(f0_valido) > 0 else 0
    )

    resultados.append({
        "arquivo": linha["arquivo"],
        "classe": linha["classe"],
        "grupo_original": linha["arquivo_original"],
        "transcricao_whisper": texto,
        "duracao": duracao,
        "energia_media": energia_media,
        "proporcao_pausas": proporcao_pausas,
        "ritmo_palavras_minuto": ritmo,
        "tom_medio_hz": tom_medio,
        "tom_desvio_hz": tom_desvio
    })

df_audio = pd.DataFrame(resultados)

df_audio.to_csv(
    "/content/caracteristicas_audio.csv",
    index=False,
    encoding="utf-8"
)

df_audio.head()

100%|█████████████████████████████████████| 72.1M/72.1M [00:01<00:00, 63.0MiB/s]


  0%|          | 0/1200 [00:00<?, ?it/s]

,arquivo,classe,grupo_original,transcricao_whisper,duracao,energia_media,proporcao_pausas,ritmo_palavras_minuto,tom_medio_hz,tom_desvio_hz
0,ansiedade_hesitacao_000_original.wav,ansiedade_hesitacao,ansiedade_hesitacao_000.wav,"Um, preciso de ajuda para ficar mais tranquilo.",4.183562,0.084544,0.242751,114.734751,272.764959,51.259933
1,ansiedade_hesitacao_000_lento.wav,ansiedade_hesitacao,ansiedade_hesitacao_000.wav,Hem Meu Team É como Local de Doves Gostid meanto,4.547375,0.051438,0.218890,131.944254,265.488003,57.305810
2,ansiedade_hesitacao_000_rapido.wav,ansiedade_hesitacao,ansiedade_hesitacao_000.wav,Um... esse isaja ajuda a partir acho mais umasa!,3.873687,0.061310,0.223479,139.402056,268.900522,56.117284
3,ansiedade_hesitacao_000_ruido.wav,ansiedade_hesitacao,ansiedade_hesitacao_000.wav,Um preciso de ajuda para ficar mais tranquilo.,4.183562,0.085294,0.235102,114.734751,272.694554,51.164818
4,ansiedade_hesitacao_001_original.wav,ansiedade_hesitacao,ansiedade_hesitacao_001.wav,"Bem, preciso de ajuda para ficar mais tranquilo.",4.128188,0.077902,0.232593,116.273788,270.199429,57.908848


In [16]:
import numpy as np
import pandas as pd
import librosa
import joblib

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from google.colab import drive

colunas = [
    "duracao",
    "energia_media",
    "proporcao_pausas",
    "ritmo_palavras_minuto",
    "tom_medio_hz",
    "tom_desvio_hz"
]

X = df_audio[colunas].fillna(0)
y = df_audio["classe"]
grupos = df_audio["grupo_original"]

divisor = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
indice_treino, indice_teste = next(divisor.split(X, y, groups=grupos))

modelo_avaliacao = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")
modelo_avaliacao.fit(X.iloc[indice_treino], y.iloc[indice_treino])
previsoes = modelo_avaliacao.predict(X.iloc[indice_teste])
print(classification_report(y.iloc[indice_teste], previsoes))

# Treina o modelo final com todos os dados após a avaliação.
modelo_audio = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")
modelo_audio.fit(X, y)

drive.mount("/content/drive")
CAMINHO_MODELO = "/content/drive/MyDrive/modelo_audio.joblib"
joblib.dump({"modelo": modelo_audio, "colunas": colunas}, CAMINHO_MODELO)
print(f"Modelo salvo em: {CAMINHO_MODELO}")


                     precision    recall  f1-score   support

ansiedade_hesitacao       0.93      0.84      0.88        88
             neutro       0.81      0.91      0.85        64
           tristeza       1.00      1.00      1.00        88

           accuracy                           0.92       240
          macro avg       0.91      0.92      0.91       240
       weighted avg       0.92      0.92      0.92       240

Mounted at /content/drive
Modelo salvo em: /content/drive/MyDrive/modelo_audio.joblib


In [17]:
def classificar_audio(caminho_audio):
    audio, sr = librosa.load(
        str(caminho_audio),
        sr=16000,
        mono=True
    )

    duracao = librosa.get_duration(y=audio, sr=sr)
    energia_media = float(np.mean(librosa.feature.rms(y=audio)[0]))

    intervalos = librosa.effects.split(audio, top_db=30)
    tempo_sonoro = sum(
        (fim - inicio) / sr
        for inicio, fim in intervalos
    )
    proporcao_pausas = (
        max(duracao - tempo_sonoro, 0) / duracao
        if duracao > 0 else 0
    )

    texto = modelo_whisper.transcribe(
        str(caminho_audio),
        language="pt",
        fp16=False
    )["text"].strip()

    ritmo = (
        len(texto.split()) / (duracao / 60)
        if duracao > 0 else 0
    )

    f0, _, _ = librosa.pyin(
        audio,
        fmin=70,
        fmax=500,
        sr=sr
    )
    f0_valido = f0[~np.isnan(f0)]

    tom_medio = float(np.mean(f0_valido)) if len(f0_valido) else 0.0
    tom_desvio = float(np.std(f0_valido)) if len(f0_valido) else 0.0

    amostra = pd.DataFrame([{
        "duracao": duracao,
        "energia_media": energia_media,
        "proporcao_pausas": proporcao_pausas,
        "ritmo_palavras_minuto": ritmo,
        "tom_medio_hz": tom_medio,
        "tom_desvio_hz": tom_desvio
    }], columns=colunas).fillna(0)

    probabilidades = modelo_audio.predict_proba(amostra)[0]
    indice = int(np.argmax(probabilidades))

    return (
        str(modelo_audio.classes_[indice]),
        float(probabilidades[indice])
    )

In [22]:
from pathlib import Path

PASTA_AUDIOS = Path("/content/dataset_audio_emocoes_1200")
arquivo_teste = next(PASTA_AUDIOS.rglob("ansiedade_hesitacao_004_original.wav"))
classe, confianca = classificar_audio(str(arquivo_teste))

print(arquivo_teste.name, classe, confianca)


ansiedade_hesitacao_004.wav ansiedade_hesitacao 0.93


In [25]:
from pathlib import Path
import pandas as pd
import soundfile as sf

# Pasta para onde o ZIP foi extraído
PASTA_AUDIOS = Path("/content/dataset_audio_emocoes_1200")

# Localiza todos os WAVs, inclusive os que estiverem em subpastas
arquivos_audio = sorted(PASTA_AUDIOS.rglob("*.wav"))

if not arquivos_audio:
    raise FileNotFoundError(
        f"Nenhum arquivo WAV encontrado em {PASTA_AUDIOS}. "
        "Confira se o ZIP foi extraído para essa pasta."
    )

print(f"WAVs encontrados: {len(arquivos_audio)}")

# Verifica quais arquivos podem ser lidos como áudio
validos = []
invalidos = []

for arquivo in arquivos_audio:
    try:
        info = sf.info(str(arquivo))
        if info.frames > 0 and info.samplerate > 0:
            validos.append(arquivo)
        else:
            invalidos.append((arquivo, "sem frames de áudio"))
    except Exception as erro:
        invalidos.append((arquivo, str(erro)))

print(f"WAVs válidos: {len(validos)}")
print(f"WAVs inválidos: {len(invalidos)}")

# Remove arquivos inválidos somente da cópia temporária em /content.
# O ZIP original no Drive não é alterado.
for arquivo, motivo in invalidos:
    try:
        arquivo.resolve().relative_to(Path("/content").resolve())
        arquivo.unlink()
        print(f"Removido da cópia extraída: {arquivo.name}")
    except ValueError:
        print(f"Preservado por estar fora de /content: {arquivo}")
    except OSError as erro:
        print(f"Não foi possível remover {arquivo.name}: {erro}")

# Classifica cada WAV válido; erros individuais não interrompem o lote
resultados = []
erros_classificacao = []

for i, arquivo in enumerate(validos, start=1):
    try:
        classe, confianca = classificar_audio(str(arquivo))
        resultados.append({
            "arquivo": str(arquivo.relative_to(PASTA_AUDIOS)),
            "classe_audio": classe,
            "confianca_audio": confianca
        })
    except Exception as erro:
        erros_classificacao.append({
            "arquivo": str(arquivo.relative_to(PASTA_AUDIOS)),
            "erro": str(erro)
        })

    if i % 25 == 0:
        print(f"Processados: {i}/{len(validos)}")

# Salva os resultados e o relatório de falhas no Google Drive
CAMINHO_RESULTADOS = "/content/drive/MyDrive/resultado_audio.csv"
CAMINHO_ERROS = "/content/drive/MyDrive/erros_classificacao_audio.csv"

pd.DataFrame(resultados).to_csv(
    CAMINHO_RESULTADOS,
    index=False,
    encoding="utf-8"
)

pd.DataFrame(erros_classificacao).to_csv(
    CAMINHO_ERROS,
    index=False,
    encoding="utf-8"
)

print(f"\nÁudios classificados: {len(resultados)}")
print(f"Falhas durante classificação: {len(erros_classificacao)}")
print(f"Resultados salvos em: {CAMINHO_RESULTADOS}")
print(f"Relatório de falhas salvo em: {CAMINHO_ERROS}")

df_resultados = pd.DataFrame(resultados)
df_resultados.head()

WAVs encontrados: 1200
WAVs válidos: 1200
WAVs inválidos: 0
Processados: 25/1200
Processados: 50/1200
Processados: 75/1200
Processados: 100/1200
Processados: 125/1200
Processados: 150/1200
Processados: 175/1200
Processados: 200/1200
Processados: 225/1200
Processados: 250/1200
Processados: 275/1200
Processados: 300/1200
Processados: 325/1200
Processados: 350/1200
Processados: 375/1200
Processados: 400/1200
Processados: 425/1200
Processados: 450/1200
Processados: 475/1200
Processados: 500/1200
Processados: 525/1200
Processados: 550/1200
Processados: 575/1200
Processados: 600/1200
Processados: 625/1200
Processados: 650/1200
Processados: 675/1200
Processados: 700/1200
Processados: 725/1200
Processados: 750/1200
Processados: 775/1200
Processados: 800/1200
Processados: 825/1200
Processados: 850/1200
Processados: 875/1200
Processados: 900/1200
Processados: 925/1200
Processados: 950/1200
Processados: 975/1200
Processados: 1000/1200
Processados: 1025/1200
Processados: 1050/1200
Processados: 107

,arquivo,classe_audio,confianca_audio
0,ansiedade_hesitacao_000_lento.wav,ansiedade_hesitacao,0.865
1,ansiedade_hesitacao_000_original.wav,ansiedade_hesitacao,0.975
2,ansiedade_hesitacao_000_rapido.wav,ansiedade_hesitacao,0.820
3,ansiedade_hesitacao_000_ruido.wav,ansiedade_hesitacao,0.955
4,ansiedade_hesitacao_001_lento.wav,ansiedade_hesitacao,0.910
